# Task 3 — Kafka Topic Design

## Approach and reasoning

Four topics carry the four event categories required by the lab:

| Topic | Message key | Partitions | `cleanup.policy` | Rationale |
|---|---|---|---|---|
| `cpg.nodes` | node id | 6 | compact | high volume, parallel sink consumption |
| `cpg.edges` | edge id | 6 | compact | high volume |
| `cpg.metadata` | file id | 1 | compact | one message per file, ordering trivial |
| `cpg.errors` | file id | 1 | delete, 7 days | diagnostics, not state |

### Why the key is the stable element id
Keying by the element's stable id gives us three properties at once:

1. **Ordering** — all versions of one element hash to the same partition, so a
   later version can never be overtaken by an earlier one.
2. **Compaction safety** — with `cleanup.policy=compact`, Kafka retains only the
   latest value per key. The topic therefore tracks the *current* graph rather
   than the cumulative event history, and its size stays bounded across replays.
3. **Alignment with the sink** — compaction semantics ("latest per key wins")
   mirror the `MERGE` semantics used in Neo4j, so the two layers agree.

### Message envelope
Every message carries `schema_version` (forward compatibility, required by the
lab) and `event_time` as ISO-8601 UTC (required by the lab), plus `file_id` and
`file_hash` so downstream consumers can perform generation-based cleanup.

The four contracts are formalised as JSON Schema in `schemas/`.

In [1]:
import os, json
os.chdir("..")
for name in ["node", "edge", "metadata", "error"]:
    s = json.load(open(f"schemas/{name}_event.schema.json"))
    print(f"{name:9} required: {s['required']}")

node      required: ['schema_version', 'event_type', 'event_time', 'file_id', 'file_hash', 'node']
edge      required: ['schema_version', 'event_type', 'event_time', 'file_id', 'file_hash', 'edge']
metadata  required: ['schema_version', 'event_type', 'event_time', 'file_id', 'file_hash', 'rel_path']
error     required: ['schema_version', 'event_type', 'event_time', 'file_id', 'rel_path', 'error_type', 'message']


### Validate a real produced event against its schema

In [2]:
import json, jsonschema
schema = json.load(open("schemas/node_event.schema.json"))
event = json.loads(open("reports/events/samples/cpg.nodes.sample.jsonl").readline())["value"]
jsonschema.validate(event, schema)
print("node event VALID against schema")
print("schema_version:", event["schema_version"], "| event_time:", event["event_time"])

node event VALID against schema
schema_version: 1.0.0 | event_time: 2026-07-22T18:18:45.020019+00:00


### Create the topics on the live cluster

> **Run this cell against the running Docker stack.** It requires
> `docker compose up -d` to have completed and the broker healthcheck to pass.

In [3]:
!python src/kafka_setup/create_topics.py --bootstrap localhost:9092

skip     cpg.nodes: [Error 36] TopicAlreadyExistsError: Request 'CreateTopicsRequest_v3(create_topic_requests=[(topic='cpg.nodes', num_partitions=6, replication_factor=1, replica_assignment=[], configs=[(config_key='cleanup.policy', config_value='compact')])], timeout=30000, validate_only=False)' failed with response 'CreateTopicsResponse_v3(throttle_time_ms=0, topic_errors=[(topic='cpg.nodes', error_code=36, error_message="Topic 'cpg.nodes' already exists.")])'.
skip     cpg.edges: [Error 36] TopicAlreadyExistsError: Request 'CreateTopicsRequest_v3(create_topic_requests=[(topic='cpg.edges', num_partitions=6, replication_factor=1, replica_assignment=[], configs=[(config_key='cleanup.policy', config_value='compact')])], timeout=30000, validate_only=False)' failed with response 'CreateTopicsResponse_v3(throttle_time_ms=0, topic_errors=[(topic='cpg.edges', error_code=36, error_message="Topic 'cpg.edges' already exists.")])'.
skip     cpg.metadata: [Error 36] TopicAlreadyExistsError: Reque

### Confirm the topics exist and inspect their configuration

In [4]:
!docker exec broker kafka-topics --bootstrap-server localhost:9092 --list
!docker exec broker kafka-topics --bootstrap-server localhost:9092 --describe --topic cpg.nodes

__consumer_offsets
_connect-configs
_connect-offsets
_connect-status
cpg.edges
cpg.edges.dlq
cpg.errors
cpg.metadata
cpg.nodes
cpg.nodes.dlq


Topic: cpg.nodes	TopicId: HG_5LQ7CSQi1vSnYDhXyiA	PartitionCount: 6	ReplicationFactor: 1	Configs: cleanup.policy=compact
	Topic: cpg.nodes	Partition: 0	Leader: 1	Replicas: 1	Isr: 1
	Topic: cpg.nodes	Partition: 1	Leader: 1	Replicas: 1	Isr: 1
	Topic: cpg.nodes	Partition: 2	Leader: 1	Replicas: 1	Isr: 1
	Topic: cpg.nodes	Partition: 3	Leader: 1	Replicas: 1	Isr: 1
	Topic: cpg.nodes	Partition: 4	Leader: 1	Replicas: 1	Isr: 1
	Topic: cpg.nodes	Partition: 5	Leader: 1	Replicas: 1	Isr: 1


### Sample live messages from the topic (after the parser has run)

In [5]:
!docker exec broker kafka-console-consumer --bootstrap-server localhost:9092 \
    --topic cpg.nodes --from-beginning --max-messages 2 --timeout-ms 10000

{"schema_version": "1.0.0", "event_type": "node", "event_time": "2026-07-23T07:10:51.618848+00:00", "file_id": "e874a4257ccdb966", "rel_path": "docs/combine_docs.py", "file_hash": "44654054ebb70e05bd2347ed2e38fcd3dd1b203364593391dabc1769ab26f1ee", "node": {"id": "a4b38595cc807f1713da5b9a", "type": "Import", "label": "", "code": "import yaml", "start_line": 6, "end_line": 6, "scope_id": "b295fb53090cf376624df61f"}}
{"schema_version": "1.0.0", "event_type": "node", "event_time": "2026-07-23T07:10:51.618944+00:00", "file_id": "e874a4257ccdb966", "rel_path": "docs/combine_docs.py", "file_hash": "44654054ebb70e05bd2347ed2e38fcd3dd1b203364593391dabc1769ab26f1ee", "node": {"id": "3661af8ff9fd7b10ae1814ab", "type": "alias", "label": "yaml", "code": "import yaml", "start_line": 6, "end_line": 6, "scope_id": "b295fb53090cf376624df61f"}}


Processed a total of 2 messages


## Reflection

**What failed on this machine.** The parser crashed at producer construction with
`AssertionError: Unrecognized configs: {'enable_idempotence': True}`. On
Python 3.12 this project uses `kafka-python-ng` 2.2.3 (upstream `kafka-python`
does not import on 3.12), and that fork rejects the `enable_idempotence` flag
instead of ignoring it. The fix in `parser_service.py` tries the flag and falls
back to a plain `acks="all"` producer. This is safe here: the pipeline's
idempotency comes from structural ids plus `MERGE`/upsert in the sinks, so a
broker-side retry duplicate is absorbed exactly like a full replay
(TROUBLESHOOTING.md §11).

**What worked.** Topic creation is idempotent — re-running the pipeline hit
`TopicAlreadyExistsError` on all four topics and skipped them cleanly. Log
compaction was the non-obvious win: because keys are stable ids, a replayed file
overwrites its own messages instead of appending, so topic size tracks the
current graph rather than growing with every replay.

**What to watch.** Partition count is a one-way door in Kafka — increasing it
later re-hashes keys and breaks per-element ordering. We picked 6 for the
high-volume topics up front rather than starting at 1.
